## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import StratifiedKFold

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Parameter

In [4]:
current_working_directory = os.getcwd()
print(current_working_directory)

/Users/rapido/analytics/customer/Ads-Monetisation/captian_dispatch


In [5]:
local_datasets = '/Users/rapido/local-datasets/customer/ads-monetisation/captian_dispatch_exp/banner_monetisation/'

In [6]:
## Parameter 
start_date = '20240210'
end_date = '20240218'
cities = ['Bangalore', 'Jaipur']

In [7]:
# Convert the lists to comma-separated strings
city_str = "', '".join(cities)
print(city_str)

Bangalore', 'Jaipur


## Datasets

In [16]:
# Loop through each date
current_date = start_date
print('execution started..')
while current_date <= end_date:
    print('running.... ' + current_date)
    banner_monetisation = f"""
                
        SELECT 
            yyyymmdd,
            eventname,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.currentCity') city,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.userId') userId,
            JSON_EXTRACT_SCALAR(profile, '$.profile.platform') platform,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.placement') placement,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.action') action,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.sub_type') sub_type,
            epoch,
            eventProps
        FROM 
            raw_internal.clevertap_customer_events_v2 
        WHERE 
            yyyymmdd = '{current_date}'
            AND eventname = 'banner_monetisation'
            AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.currentCity') IN ('{city_str}')
            AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.action') IN ('view', 'click')
            AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.placement') IN ( 'CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd', 'CaptainSearchScreenSlot:d1d2227b-92be-4318-b765-89405bcd1787'   )
        
        GROUP BY 1,2,3,4,5,6,7,8,9,10
    """

    df_banner_monetisation = pd.read_sql(banner_monetisation, connection)
    
    output_filename = f"captian_dispatch_exp_test_{current_date}.csv"    
    df_banner_monetisation.to_csv(local_datasets + output_filename, index=False)
    current_date = (pd.to_datetime(current_date) + pd.DateOffset(days=1)).strftime('%Y%m%d')
print('execution completed..')

execution started..
running.... 20240210
running.... 20240211
running.... 20240212
running.... 20240213
running.... 20240214
running.... 20240215
running.... 20240216
running.... 20240217
running.... 20240218
execution completed..


In [17]:
dfs = []

for filename in os.listdir(local_datasets):
    if filename.endswith(".csv"):
        dfs.append(pd.read_csv(os.path.join(local_datasets, filename)))

combined_df = pd.concat(dfs, ignore_index=True)

In [18]:
combined_df

,yyyymmdd,eventname,city,userId,platform,placement,action,sub_type,epoch,eventProps
0,20240213,banner_monetisation,Jaipur,62a9488317af9a44d857e820,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,view,nativeBanner,1707789457000,"{""eventProps"":{""GAM_Banner_constant"":""[X-LARGE]"",""GAM_Banner_size"":""[null]"",""SlotType"":""STATIC"",""action"":""view"",""app_variant"":""v8"",""card_rank"":0.0,""carousel_auto_scroll_flag"":""false"",""carousel_total_card_count"":1.0,""cleverTapId"":""__de9aecdbb66d4fe9aaabe3c56955614d"",""currentCity"":""Jaipur"",""currentCluster"":""Railway_Station_JAI"",""epoch"":""1707789457784"",""eventSource"":""App"",""hhmmss"":""072737"",""latitude"":26.919756,""locationAccuracy"":12.595,""longitude"":75.78841,""occurred_date"":""Tue Feb 13 2024 07:27:37 am"",""order_status"":"""",""placement"":""CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd"",""screen_name"":""CaptainSearchScreen"",""slotValue"":"""",""sub_type"":""nativeBanner"",""type"":""GAMBanner"",""userId"":""62a9488317af9a44d857e820"",""yyyymmdd"":""20240213"",""CTAppVersion"":""8.16.0"",""CTLatitude"":26.920305,""CTLongitude"":75.78879,""CTSessionId"":1.707789326E9,""CTSource"":""Mobile""}}"
1,20240213,banner_monetisation,Jaipur,64bcfdb87c50004f3a92e865,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,view,nativeBanner,1707806187000,"{""eventProps"":{""GAM_Banner_constant"":""[X-LARGE]"",""GAM_Banner_size"":""[null]"",""SlotType"":""STATIC"",""action"":""view"",""app_variant"":""v8"",""card_rank"":0.0,""carousel_total_card_count"":1.0,""cleverTapId"":""__d35190039dd14c4a948a48c71e90d3b5"",""currentCity"":""Jaipur"",""currentCluster"":""Sudarshanpura_JAI"",""epoch"":""1707806187297"",""eventSource"":""App"",""hhmmss"":""120627"",""latitude"":26.887627,""locationAccuracy"":3.9,""longitude"":75.79227,""occurred_date"":""Tue Feb 13 2024 12:06:27 pm"",""order_status"":"""",""placement"":""CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd"",""screen_name"":""CaptainSearchScreen"",""slotValue"":"""",""sub_type"":""nativeBanner"",""type"":""GAMBanner"",""userId"":""64bcfdb87c50004f3a92e865"",""yyyymmdd"":""20240213"",""CTAppVersion"":""8.15.0"",""CTLatitude"":26.887627,""CTLongitude"":75.79227,""CTSessionId"":1.707806133E9,""CTSource"":""Mobile""}}"
2,20240213,banner_monetisation,Jaipur,612466b2342bfe6f3fed83ec,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,view,nativeBanner,1707820915000,"{""eventProps"":{""GAM_Banner_constant"":""[X-LARGE]"",""GAM_Banner_size"":""[null]"",""SlotType"":""STATIC"",""action"":""view"",""app_variant"":""v8"",""card_rank"":0.0,""carousel_auto_scroll_flag"":""false"",""carousel_total_card_count"":1.0,""cleverTapId"":""__6728abd3266243a8bc8ac8473253b0fb"",""currentCity"":""Jaipur"",""currentCluster"":""Narayan Vihar_JAI"",""epoch"":""1707820915288"",""eventSource"":""App"",""hhmmss"":""161155"",""latitude"":26.874426,""locationAccuracy"":11.405,""longitude"":75.74062,""occurred_date"":""Tue Feb 13 2024 16:11:55 pm"",""order_status"":"""",""placement"":""CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd"",""screen_name"":""CaptainSearchScreen"",""slotValue"":"""",""sub_type"":""nativeBanner"",""type"":""GAMBanner"",""userId"":""612466b2342bfe6f3fed83ec"",""yyyymmdd"":""20240213"",""CTAppVersion"":""8.16.0"",""CTLatitude"":26.87545,""CTLongitude"":75.73939,""CTSessionId"":1.707820682E9,""CTSource"":""Mobile""}}"
3,20240213,banner_monetisation,Jaipur,62c5806917af9ab49a6b9070,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,view,nativeBanner,1707815321000,"{""eventProps"":{""GAM_Banner_constant"":""[X-LARGE]"",""GAM_Banner_size"":""[null]"",""SlotType"":""STATIC"",""action"":""view"",""app_variant"":""v8"",""card_rank"":0.0,""carousel_auto_scroll_flag"":""false"",""carousel_total_card_count"":1.0,""cleverTapId"":""__28c3c470c5f940f1a0fff724bb4e246a"",""currentCity"":""Jaipur"",""currentCluster"":""Shive Nath

In [20]:
combined_df\
.groupby(['yyyymmdd','eventname','city', 'platform', 'placement', 'sub_type', 'action'])\
.agg(user_count = pd.NamedAgg('userId', 'nunique'))\
.reset_index()

,yyyymmdd,eventname,city,platform,placement,sub_type,action,user_count
0,20240212,banner_monetisation,Bangalore,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,nativeBanner,click,10
1,20240212,banner_monetisation,Bangalore,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,nativeBanner,view,1194
2,20240212,banner_monetisation,Jaipur,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,nativeBanner,click,20
3,20240212,banner_monetisation,Jaipur,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,nativeBanner,view,1196
4,20240213,banner_monetisation,Bangalore,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,nativeBanner,click,37
5,20240213,banner_monetisation,Bangalore,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,nativeBanner,view,3613
6,20240213,banner_monetisation,Jaipur,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,nativeBanner,click,41
7,20240213,banner_monetisation,Jaipur,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,nativeBanner,view,3668
8,20240214,banner_monetisation,Bangalore,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,nativeBanner,click,44
9,20240214,banner_monetisation,Bangalore,Android,CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd,nativeBanner,view,3174


### Quick Funnel 